In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install mne --quiet

import os
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d
from mne.time_frequency import morlet
import glob
from scipy.signal import convolve
from scipy.io import loadmat
from mne import *

In [ ]:
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli/'
# wav_files = sorted(glob.glob(os.path.join(root_dir, '*.wav'))) # all
wav_files = sorted([
    f for f in glob.glob(os.path.join(root_dir, '*.wav'))
    if f.endswith('_N.wav') or f.endswith('_E.wav')
])

In [ ]:
smooth_ms = 10

for wav_path in wav_files:
    # load audio
    fs, data = wav.read(wav_path)
    if data.ndim > 1:
        # RMS
        data = np.sqrt(np.mean(data.astype(np.float32)**2, axis=1))
    else:
        data = data.astype(np.float32)

    data = data / np.max(np.abs(data)) # normalize

    # compute amplitude envelope via Hilbert transform
    analytic = hilbert(data)
    envelope = np.abs(analytic)

    # smooth the envelope with Gaussian filter
    sigma = (smooth_ms / 1000) * fs  # convert ms to samples
    envelope_smooth = gaussian_filter1d(envelope, sigma)

    # compute first derivative of envelope
    dt = 1.0 / fs
    envelope_derivative = np.gradient(envelope_smooth, dt)

    # time axis
    t = np.arange(len(data)) / fs

    # plot waveform and smoothed envelope
    plt.figure(figsize=(12, 5))
    plt.plot(t, data, linewidth=0.5, label='Waveform')
    plt.plot(t, envelope_smooth, linewidth=2, label='Envelope')
    plt.title('Broadband Amplitude Envelope — ' + os.path.basename(wav_path))
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

    # plot first derivative of envelope
    plt.figure(figsize=(12, 4))
    plt.plot(t, envelope_derivative, color='orange', label='Envelope Derivative')
    plt.xlabel('Time [s]')
    plt.ylabel('dEnvelope/dt')
    plt.title('First Derivative of Amplitude Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# read and show waveform of SR2_N.wav
fs, data = wav.read('/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli/SR2_N.wav')
plt.figure(figsize=(12, 5))
plt.plot(data)

In [ ]:
freqs = np.linspace(100, 10000, 200)  # 20 frequencies from 2 to 40 Hz
# freqs = np.array([60])
n_cycles = 7                    # Number of cycles per wavelet
smooth_ms = 10

for wav_path in wav_files:
    fs, data = wav.read(wav_path)
    if data.ndim > 1:
        data = np.sqrt(np.mean(data.astype(np.float32)**2, axis=1))
    else:
        data = data.astype(np.float32)

    data = data / np.max(np.abs(data)) # normalize

    # Compute wavelets
    sfreq = fs

    # Create Morlet wavelets and apply
    wavelet_output = morlet(sfreq=sfreq, freqs=freqs, n_cycles=n_cycles)

    # Convolve each wavelet with signal and extract amplitude
    power = np.empty((len(freqs), len(data)))
    for i, w in enumerate(wavelet_output):
      conv = convolve(data, w, mode='same')  # shape will match data
      power[i, :] = np.abs(conv)

    # Average amplitude across frequencies
    envelope_wavelet_avg = power.mean(axis=0)

    # Smooth the envelope
    sigma = (smooth_ms / 1000.0) * fs
    envelope_smooth = gaussian_filter1d(envelope_wavelet_avg, sigma)

    # First derivative of smoothed envelope
    dt = 1.0 / fs
    envelope_derivative = np.gradient(envelope_smooth, dt)

    # Time axis
    t = np.arange(len(data)) / fs

    # Plot envelope and waveform
    plt.figure(figsize=(12, 5))
    plt.plot(t, data, linewidth=0.5, label='Waveform')
    plt.plot(t, envelope_smooth, linewidth=2, label='Wavelet Envelope')
    plt.title('Wavelet-based Envelope — ' + os.path.basename(wav_path))
    plt.xlabel('Time [s]')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Plot derivative of envelope
    plt.figure(figsize=(12, 4))
    plt.plot(t, envelope_derivative, color='orange', label='Envelope Derivative')
    plt.xlabel('Time [s]')
    plt.ylabel('dEnvelope/dt')
    plt.title('First Derivative of Wavelet Envelope — ' + os.path.basename(wav_path))
    plt.legend()
    plt.tight_layout()
    plt.show()